In [17]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
import os
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import matplotlib.colors as mcolors
import matplotlib.markers as mmarkers
from matplotlib.lines import Line2D
import seaborn as sns
import scipy

from compcats.localizer import localizer
from compcats.datasets.fmri import nsd
from compcats.utils import model_utils

import main_analysis as ma
import paths

# Plot settings
import matplotlib as mpl
from matplotlib import rcParams
rcParams.update({'figure.autolayout': False})
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Avenir'

RESULTS_DIR = 'results'
MODELS_DIR = 'models'
PLOTS_DIR = f'{RESULTS_DIR}/plots/preprint/section4'
os.makedirs(PLOTS_DIR, exist_ok=True)

NSD_DIR = paths.NSD_DIR
GRILLSPECTOR_LOCALIZER_DIR = paths.GRILLSPECTOR_LOCALIZER_DIR
KONKLE_LOCALIZER_DIR = paths.KONKLE_LOCALIZER_DIR
KANWISHER_LOCALIZER_DIR = paths.KANWISHER_LOCALIZER_DIR


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Utils

def get_shades(hex_color, n=5, lighten_limit=0.25, darken_limit=0.75):
    """
    Generate n shades around a base hex_color.
    Base sits in the middle, lighter tints above, darker shades below.
    
    lighten_limit: how much lighter the lightest color can get (0-1)
    darken_limit: how much darker the darkest color can get (0-1)
    """
    rgb = np.array(mcolors.to_rgb(hex_color))
    colors = []
    mid = (n - 1) / 2
    
    for i in range(n):
        factor = (i - mid) / mid  # ranges -1..1
        if factor < 0:
            # lighter: move towards white but limited
            new_rgb = rgb + (1 - rgb) * (-factor * lighten_limit)
        else:
            # darker: move towards black but limited
            new_rgb = rgb * (1 - factor * darken_limit)
        colors.append(np.clip(new_rgb, 0, 1))
    return colors

# Main figure (TODO)

## Fig 4c: Functional localizer

In [18]:
tthreshold = 7
frois = ['FFA', 'EBA', 'PPA']
model_localizers = ['grillspector', 'kanwisher']

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/localizer_effect_correlations_nsd515.csv'

In [19]:
savefig = True

plot_dir = f'{PLOTS_DIR}/main_figure'
os.makedirs(plot_dir, exist_ok=True)

subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
modelsubject_correlations_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

# frois = subjectsubject_correlations_csv['fmri_froi'].unique()
subjects = range(1, 9) # subject IDs are 1-indexed in the CSV
models = modelsubject_correlations_csv['model_name'].unique()

hh_color = '#fde0c5' # Light orange for Subject-Subject correlations
mh_color = '#944e1f'
# localizer_color_dict = {
#     'grillspector': '#ffffff',  # Pure white for Grill-Spector localizer
#     'kanwisher': '#B07FA0',            # Light red for Kanwisher localizer
#     'nsd': '#B07FA0'       # Light red for NSD localizer CE8F9C
# }
colors = [hh_color, mh_color, mh_color]

metrics = ['normed_pearsonr', 'normed_rdmr']
metric_ylims = {
    'normed_pearsonr': (-0.25, 1.15),
    'normed_rdmr': (-0.1, 1.4)
}

# fig, ax = plt.figure(figsize=(5, 5), dpi=300)
# fig, axs = plt.subplots(1, len(frois), figsize=(5*len(frois), 5), dpi=300)
fig, axs = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

stats_df = []

for midx, metric in enumerate(metrics):
    for fidx, froi in enumerate(frois):
        hh_medians = []
        for subject in subjects:
            subjectsubject_froi = subjectsubject_correlations_csv[
                (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                (subjectsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                (
                    (subjectsubject_correlations_csv['subject1'] == subject) | 
                    (subjectsubject_correlations_csv['subject2'] == subject)
                )
            ][metric]
            hh_medians.append(subjectsubject_froi.median())
        hh_medians = np.array(hh_medians).flatten()

        mh_medians, mh_sems = {}, {}
        for model in models:
            for model_localizer in model_localizers:
                if model_localizer not in mh_medians:
                    mh_medians[model_localizer] = []
                    mh_sems[model_localizer] = []
                modelsubject_froi = modelsubject_correlations_csv[
                    (modelsubject_correlations_csv['fmri_froi'] == froi) &
                    (modelsubject_correlations_csv['model_name'] == model) &
                    (modelsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_localizer'] == model_localizer)
                ]
                mh_medians[model_localizer].append(modelsubject_froi[metric].median())
                mh_sems[model_localizer].append(modelsubject_froi[metric].sem())
        gs_mh_medians = np.array(mh_medians['grillspector']).flatten()
        gs_mh_sems = np.array(mh_sems['grillspector']).flatten()
        kw_mh_medians = np.array(mh_medians['kanwisher']).flatten()
        kw_mh_sems = np.array(mh_sems['kanwisher']).flatten()

        jitter_scale = 0.01 #0.05
        scatter_pad = 0 # 0.05
        alpha = 0.75 # 0.5
        markersize = 15 # 25

        bar_positions = np.array([0, 0.5, 1.0])  # Positions for the boxplot bars

        # y=0 line
        axs[midx, fidx].axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=0.75, zorder=0)

        # Box plot
        box_data = [hh_medians, gs_mh_medians, kw_mh_medians]
        linewidth = 0.75
        barwidth = 0.25 # 0.35
        boxes = axs[midx, fidx].boxplot(
                    box_data, positions=bar_positions, patch_artist=True, 
                    widths=barwidth, showfliers=False,
                    boxprops={'color': 'black', 'linewidth': linewidth},
                    whiskerprops=dict(linewidth=linewidth),
                    capprops=dict(linewidth=linewidth),
                    medianprops={'color': 'black', 'linewidth': linewidth},
                    flierprops=dict(marker='o', markersize=5, linewidth=linewidth)
                )
        for i, data in enumerate(box_data):
            med = np.median(data)
            # Manually draw a longer median line
            x = bar_positions[i]  # X-position of the boxplot
            # 0.225
            axs[midx, fidx].plot([x - 0.175, x + 0.175], [med, med], color='black', linewidth=linewidth, solid_capstyle='butt')
        
        # Color the boxes according to category colors
        for box, color in zip(boxes['boxes'], colors):
            box.set(facecolor=color)

        x_subjectsubject = np.zeros(len(hh_medians)) + np.random.normal(0, jitter_scale, len(hh_medians)) 
        x_modelsubject_gs = np.ones(len(gs_mh_medians))*bar_positions[1] + np.random.normal(0, jitter_scale, len(gs_mh_medians))
        x_modelsubject_kw = np.ones(len(kw_mh_medians))*bar_positions[2] + np.random.normal(0, jitter_scale, len(kw_mh_medians))
        x_subjectsubject += scatter_pad
        x_modelsubject_gs += scatter_pad
        x_modelsubject_kw += scatter_pad
        axs[midx, fidx].scatter(
            x_subjectsubject, hh_medians, color='gray', alpha=alpha, label='Subject-Subject', zorder=5,
            edgecolors='none', s=markersize
        )
        axs[midx, fidx].scatter(
            x_modelsubject_gs, gs_mh_medians, color='gray', alpha=alpha, label='Model-Subject', zorder=5,
            edgecolors='none', s=markersize
        )
        axs[midx, fidx].scatter(
            x_modelsubject_kw, kw_mh_medians, color='gray', alpha=alpha, label='Model-Subject', zorder=5,
            edgecolors='none', s=markersize
        )
        # print(f'Number of Subject-Subject correlations: {len(hh_medians)}')
        # print(f'Number of Model-Subject correlations: {len(mh_medians)}')
        
        # Conduct Welch's t-test
        hh_norm_pval = scipy.stats.shapiro(hh_medians, nan_policy='omit')[1]
        gs_mh_norm_pval = scipy.stats.shapiro(gs_mh_medians, nan_policy='omit')[1]
        kw_mh_norm_pval = scipy.stats.shapiro(kw_mh_medians, nan_policy='omit')[1]
        # Conduct unpaired t-tests for median across all pairs
        gs_mh_hh_tvalue, gs_mh_hh_pvalue = scipy.stats.ttest_ind(
            hh_medians, gs_mh_medians, equal_var=False, nan_policy='omit', alternative='greater'
        )
        kw_mh_hh_tvalue, kw_mh_hh_pvalue = scipy.stats.ttest_ind(
            hh_medians, kw_mh_medians, equal_var=False, nan_policy='omit', alternative='greater'
        )
        gs_kw_tvalue, gs_kw_pvalue = scipy.stats.ttest_ind(
            kw_mh_medians, gs_mh_medians, equal_var=False, nan_policy='omit', alternative='greater'
        )
        # Conduct Mann-Whitney U test for median across all pairs
        gs_mh_hh_mw_uvalue, gs_mh_hh_mw_pvalue = scipy.stats.mannwhitneyu(
            hh_medians, gs_mh_medians, alternative='greater', use_continuity=True, nan_policy='omit'
        )
        kw_mh_hh_mw_uvalue, kw_mh_hh_mw_pvalue = scipy.stats.mannwhitneyu(
            hh_medians, kw_mh_medians, alternative='greater', use_continuity=True, nan_policy='omit'
        )
        gs_kw_mw_uvalue, gs_kw_mw_pvalue = scipy.stats.mannwhitneyu(
            kw_mh_medians, gs_mh_medians, alternative='greater', use_continuity=True, nan_policy='omit'
        )
        # Store stats
        stats_df.append({
            'froi': froi,
            'metric': metric,
            'hh_norm_pval': hh_norm_pval,
            'gs_mh_norm_pval': gs_mh_norm_pval,
            'kw_mh_norm_pval': kw_mh_norm_pval,
            'hh>gs_mh_tvalue': gs_mh_hh_tvalue,
            'hh>gs_mh_pvalue': gs_mh_hh_pvalue,
            'hh>kw_mh_tvalue': kw_mh_hh_tvalue,
            'hh>kw_mh_pvalue': kw_mh_hh_pvalue,
            'gs_mh>kw_mh_tvalue': gs_kw_tvalue,
            'gs_mh>kw_mh_pvalue': gs_kw_pvalue,
            'hh>gs_mh_mw_uvalue': gs_mh_hh_mw_uvalue,
            'hh>gs_mh_mw_pvalue': gs_mh_hh_mw_pvalue,
            'hh>kw_mh_mw_uvalue': kw_mh_hh_mw_uvalue,
            'hh>kw_mh_mw_pvalue': kw_mh_hh_mw_pvalue,
            'gs_mh>kw_mh_mw_uvalue': gs_kw_mw_uvalue,
            'gs_mh>kw_mh_mw_pvalue': gs_kw_mw_pvalue,
        })
        
        # Write froi name in bottom left corner
        axs[midx, fidx].text(0.05, 0.25, froi, transform=axs[midx, fidx].transAxes, fontsize=12, verticalalignment='top')

        axs[midx, fidx].spines['top'].set_visible(False)
        axs[midx, fidx].spines['right'].set_visible(False)
        axs[midx, fidx].spines['left'].set_position(('outward', 5))
        axs[midx, fidx].spines['bottom'].set_position(('outward', 5))
        axs[midx, fidx].set_yticks([0, 0.25, 0.5, 0.75, 1])
        axs[midx, fidx].spines['left'].set_bounds(0, 1)
        # plt.tick_params(axis='y', labelleft=False)
        # axs[midx, fidx].set_ylim(-0.25, 1.15)
        axs[midx, fidx].set_ylim(metric_ylims[metric])
        axs[midx, fidx].spines['bottom'].set_bounds(bar_positions[0], bar_positions[-1])
        axs[midx, fidx].set_xlim(bar_positions[0]-0.25, bar_positions[-1]+0.25)
        axs[midx, fidx].tick_params(axis='x', labelbottom=False)

        if froi != 'FFA':
            axs[midx, fidx].spines['left'].set_visible(False)
            # Stop showing y-axis ticks
            axs[midx, fidx].yaxis.set_ticks([])
    
stats_df = pd.DataFrame(stats_df)
# sort df according to froi
stats_df = stats_df.sort_values(by=['froi', 'metric'])
display(stats_df)

if savefig:
    plt.savefig(f'{plot_dir}/fig4c_localizer_modelbraincorrelations.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
# plt.savefig(f'{plot_dir}/modelhuman_univariate_{froi}.pdf')
plt.close()

,froi,metric,hh_norm_pval,gs_mh_norm_pval,kw_mh_norm_pval,hh>gs_mh_tvalue,hh>gs_mh_pvalue,hh>kw_mh_tvalue,hh>kw_mh_pvalue,gs_mh>kw_mh_tvalue,gs_mh>kw_mh_pvalue,hh>gs_mh_mw_uvalue,hh>gs_mh_mw_pvalue,hh>kw_mh_mw_uvalue,hh>kw_mh_mw_pvalue,gs_mh>kw_mh_mw_uvalue,gs_mh>kw_mh_mw_pvalue
1,EBA,normed_pearsonr,0.032851,0.007751,0.080730,17.968567,7.064229e-18,17.798666,5.427660e-21,-4.209975,0.999955,280.0,0.000007,280.0,0.000007,281.0,0.999952
4,EBA,normed_rdmr,0.720108,0.065105,0.010673,21.893078,2.600972e-13,21.674366,1.820125e-15,-1.618525,0.944825,280.0,0.000007,280.0,0.000007,461.0,0.962902
0,FFA,normed_pearsonr,0.092294,0.825290,0.055484,21.758566,8.852766e-23,21.039729,1.176269e-23,-2.242719,0.985833,280.0,0.000007,280.0,0.000007,405.0,0.992721
3,FFA,normed_rdmr,0.125032,0.140225,0.402907,21.371593,3.422216e-15,22.615790,4.032513e-15,-0.940550,0.824865,280.0,0.000007,280.0,0.000007,524.0,0.852081
2,PPA,normed_pearsonr,0.943505,0.031438,0.055440,20.673206,1.462737e-20,29.302464,2.665373e-28,-19.192436,1.000000,280.0,0.000007,280.0,0.000007,0.0,1.000000
5,PPA,normed_rdmr,0.573252,0.992329,0.328930,20.922634,1.514216e-11,27.413464,6.463790e-13,-8.787866,1.000000,280.0,0.000007,280.0,0.000007,88.0,1.000000


# Supplementary figures

## Fig S5: best model across t-thresholds

In [ ]:
savefig = True

plot_dir = f'{PLOTS_DIR}/supplementary_figures'
os.makedirs(plot_dir, exist_ok=True)

tthresholds = [3, 5, 7, 9, 11]
frois = ['FFA', 'EBA', 'PPA']
model_localizer = 'grillspector'

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/tthreshold_effect_correlations_nsd515.csv'

metrics = ['normed_pearsonr', 'normed_rdmr'] 
fmri_t_color_base = '#fde0c5'
model_t_color_base = '#944e1f'
modelsubject_t_colors = get_shades(model_t_color_base, n=5)
metric_ylims = {
    'normed_pearsonr': (0.2, 0.6),
    'normed_rdmr': (0.4, 1)
}
best_model_name = 'vitb16siglip2res224'

subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
modelsubject_correlations_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

fig, axs = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

for midx, metric in enumerate(metrics):
    for fidx, froi in enumerate(frois):
        for ftidx, fmri_tthreshold in enumerate(tthresholds):
            # Get the subject-subject ceiling for the gap
            sbjsbj_corrs = np.array(subjectsubject_correlations_csv[
                    (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                    (subjectsubject_correlations_csv['fmri_tthreshold'] == fmri_tthreshold)
                ][metric])
            ceiling_median = np.median(sbjsbj_corrs)
            # Get model-subject correlations
            t_mdlsbj_medians, t_mdlsbj_sems = [], []
            for mtidx, model_tthreshold in enumerate(tthresholds):
                # For model_tthreshold calculate median and max model-subject correlation across all models
                mdlsbj_csv = modelsubject_correlations_csv[
                        (modelsubject_correlations_csv['fmri_froi'] == froi) &
                        (modelsubject_correlations_csv['model_froi'] == froi) &
                        (modelsubject_correlations_csv['fmri_tthreshold'] == fmri_tthreshold) &
                        (modelsubject_correlations_csv['model_tthreshold'] == model_tthreshold) &
                        (modelsubject_correlations_csv['model_localizer'] == model_localizer) &
                        (modelsubject_correlations_csv['model_name'] == best_model_name)
                    ]
                # get median and max of the aggregated values
                mdlsbj_median = mdlsbj_csv[metric].median()
                mdlsbj_sem = mdlsbj_csv[metric].sem() #TODO: Fix for NaNs
                t_mdlsbj_medians.append(ceiling_median - mdlsbj_median)
                t_mdlsbj_sems.append(mdlsbj_sem)
            # line plot with shaded area for this model tthreshold
            fmri_t_color = modelsubject_t_colors[ftidx]
            axs[midx, fidx].plot(
                tthresholds, t_mdlsbj_medians, '.-', color=fmri_t_color, linewidth=1, alpha=1,
                label=f'{fmri_tthreshold}' if fidx == 0 else None
            )
            axs[midx, fidx].errorbar(
                tthresholds, t_mdlsbj_medians, yerr=t_mdlsbj_sems,
                fmt='none', ecolor=fmri_t_color, elinewidth=1, alpha=0.75, capsize=0
            )
        axs[midx, fidx].set_xlabel('t-threshold\n(ANN)', fontsize=12)
        axs[midx, fidx].spines['top'].set_visible(False)
        axs[midx, fidx].spines['right'].set_visible(False)
        axs[midx, fidx].spines['left'].set_position(('outward', 5))
        axs[midx, fidx].spines['bottom'].set_position(('outward', 5))
        if metric == 'normed_pearsonr':
            axs[midx, fidx].set_yticks([0.25, 0.5])
            axs[midx, fidx].tick_params(axis='y', labelsize=10)
            axs[midx, fidx].spines['left'].set_bounds(0.25, 0.5)
            axs[midx, fidx].set_ylabel(rf'univariate', fontsize=12)
            # Write froi name in bottom left corner
            axs[midx, fidx].text(
                0.8, 0.1, froi, transform=axs[midx, fidx].transAxes,
                fontsize=16, verticalalignment='top'
            )
        elif metric == 'normed_rdmr':
            axs[midx, fidx].set_yticks([0.5, 0.75, 1])
            axs[midx, fidx].tick_params(axis='y', labelsize=10)
            axs[midx, fidx].spines['left'].set_bounds(0.5, 1)
            axs[midx, fidx].set_ylabel(rf'multivariate', fontsize=12)
            # Write froi name in bottom left corner
            axs[midx, fidx].text(
                0.1, 0.85, froi, transform=axs[midx, fidx].transAxes,
                fontsize=16, verticalalignment='top'
            )
        axs[midx, fidx].set_xticks(tthresholds)
        axs[midx, fidx].tick_params(axis='x', labelsize=10)
        # axs[midx, fidx].spines['left'].set_bounds(0, 1)
        # plt.tick_params(axis='y', labelleft=False)
        # axs[midx, fidx].set_ylim(-0.25, 1.15)
        axs[midx, fidx].set_ylim(metric_ylims[metric])
        axs[midx, fidx].spines['bottom'].set_bounds(tthresholds[0], tthresholds[-1])
        # axs[midx, fidx].set_xlim(bar_positions[0]-0.25, bar_positions[-1]+0.25)
        # axs[midx, fidx].tick_params(axis='x', labelbottom=False)

        if froi != 'FFA':
            axs[midx, fidx].spines['left'].set_visible(False)
            # Stop showing y-axis ticks
            axs[midx, fidx].yaxis.set_ticks([])
            axs[midx, fidx].set_ylabel('')

        # Plot legend
        if fidx == 1 and midx == 0:
            # axs[midx, fidx].legend(
            #     loc='upper left', fontsize=8,
            #     title='Brain t-thresholds', title_fontsize='6', frameon=False
            # )
            labels = [
                't-thresh = 3',
                't-thresh = 5',
                't-thresh = 7',
                't-thresh = 9',
                't-thresh = 11'
            ]
            handles = [
                Line2D([], [], linestyle='none', label=lab)
                for lab in labels
            ]
            leg = axs[midx, fidx].legend(
                handles=handles, frameon=False, handlelength=0, handletextpad=0,
                loc='upper left', fontsize=10, bbox_to_anchor=(0.15, 0.85)
            )
            # leg.set_title("brain", prop={'size': 14})
            axs[midx, fidx].text(
                0.14, 0.735, "brain", transform=axs[midx, fidx].transAxes, rotation=90,
                fontsize=12, verticalalignment='top', horizontalalignment='center',
                fontweight='bold',
            )
            # Color each legend text
            for text, color in zip(leg.get_texts(), modelsubject_t_colors):
                text.set_color(color)
fig.supylabel(r"$\Delta$ brain-brain , brain-ANN", fontsize=14)
plt.tight_layout()
if savefig:
    plt.savefig(f'{plot_dir}/figS5_ViTB16SigLIP2_correlations_tthresholds.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
plt.close(fig)

## Fig S6: IoU

In [ ]:
savefig = True

plot_dir = f'{PLOTS_DIR}/supplementary_figures'
os.makedirs(plot_dir, exist_ok=True)

iou_df_path = f'{RESULTS_DIR}/localizer_ious.csv'
ious_df = pd.read_csv(iou_df_path)
results_df = ious_df.copy()

# Group by model, model_froi and fmri_froi, and take median across subjects for the column froi_normed_pearsonr and froi_normed_rdmr
results_df = results_df[
    (results_df['tthreshold'] == 7)
]
results_df_bootstrapmedian = results_df.groupby(
    ['model_name', 'froi', 'model_localizer']
).agg({
    'iou': 'median',
}).reset_index()

frois = ['FFA', 'EBA', 'PPA']
t_threshold = 7
colors = [
    '#ffffff', # white for Grill-spector localizer
    '#c3ffff', # light cyan for union of both localizers
    '#ce9393'  # light red for Kanwisher localizer
]
froi_labels_dict = {
    'FFA': 'Face-selective',
    'PPA': 'Scene-selective',
    'EBA': 'Body-selective'
}

fig, axs = plt.subplots(1, len(frois), figsize=(5*len(frois), 5*1), dpi=300)

x_offset = 0
alpha = 0.5
markersize = 25
x = [0, 0.3, 0.6]

for fidx, froi in enumerate(frois):
    results_df_froi = results_df_bootstrapmedian[results_df_bootstrapmedian['froi'] == froi]
    # Get the points
    union_pts = np.array(results_df_froi[results_df_froi['model_localizer'] == 'union']['iou'].tolist())
    grillspector_pts = np.array(results_df_froi[results_df_froi['model_localizer'] == 'grillspector']['iou'].tolist())
    kanwisher_pts = np.array(results_df_froi[results_df_froi['model_localizer'] == 'kanwisher']['iou'].tolist())

    x_grillspector = (np.ones(len(grillspector_pts)) * x[0]) - x_offset
    x_union = (np.ones(len(union_pts)) * x[1]) - x_offset
    x_kanwisher = (np.ones(len(kanwisher_pts)) * x[2]) - x_offset

    # Bar plot with median and SEM
    bar_width = 0.1 #0.4
    # Create boxplot data
    box_data = [grillspector_pts, union_pts, kanwisher_pts]
    
    linewidth = 0.75
    barwidth = 0.075
    medianlineextra = 0.075
    
    # Create boxplot with specified colors
    bp = axs[fidx].boxplot(box_data, positions=x, widths=barwidth, 
                            patch_artist=True, showfliers=False,
                            boxprops={'color': 'black', 'linewidth': linewidth},
                            whiskerprops=dict(linewidth=linewidth),
                            capprops=dict(linewidth=linewidth),
                            medianprops={'color': 'black', 'linewidth': linewidth},
                            flierprops=dict(marker='o', markersize=5, linewidth=linewidth))
    
    # Set colors for each box
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
    
    # Manually draw longer median lines
    for i, data in enumerate(box_data):
        med = np.median(data)
        x_pos = x[i]
        axs[fidx].plot([x_pos - medianlineextra, x_pos + medianlineextra], [med, med], 
                       color='black', linewidth=linewidth, solid_capstyle='butt')

    axs[fidx].scatter(
        x_union, union_pts, color='gray', alpha=alpha, zorder=2,
        edgecolors='none', s=markersize
    )
    axs[fidx].scatter(
        x_grillspector, grillspector_pts, color='gray', alpha=alpha, zorder=2,
        edgecolors='none', s=markersize
    )
    axs[fidx].scatter(
        x_kanwisher, kanwisher_pts, color='gray', alpha=alpha, zorder=2,
        edgecolors='none', s=markersize
    )
    # connect points with a line
    for i in range(len(union_pts)):
        axs[fidx].plot(
            [x_grillspector[i], x_union[i], x_kanwisher[i]], [grillspector_pts[i], union_pts[i], kanwisher_pts[i]],
            color='gray', alpha=0.2, linewidth=0.5, zorder=1
        )
        if (x_grillspector[i] == 0 and x_union[i] == 0) \
            or (x_kanwisher[i] == 0 and x_union[i] == 0):
            # print the model name and froi
            model_name = results_df_froi.iloc[i]['model_name']
            print(f"Model: {model_name}, fROI: {froi}, no category-selective units found for one of the localizers.")

    # medians = [np.median(union_pts), np.median(grillspector_pts), np.median(kanwisher_pts)]
    # sems = [
    #     union_pts.std() / np.sqrt(len(union_pts)),
    #     grillspector_pts.std() / np.sqrt(len(grillspector_pts)),
    #     kanwisher_pts.std() / np.sqrt(len(kanwisher_pts))
    # ]

    # axs[fidx].bar(x, medians, yerr=sems, width=bar_width, color=['red', 'red'], alpha=0.25, capsize=0, error_kw={'linewidth': 1})
    # Stats
    # print(f"froi: {froi}, metric: {metric}, layer median: {np.median(union_pts):.3f}, froi median: {np.median(grillspector_pts):.3f}, lesion median: {np.median(kanwisher_pts):.3f}")
    # # Conduct Welch's t-test and mann whitney u test between layer and lesioned points
    # t_value, p_value = scipy.stats.ttest_rel(union_pts, kanwisher_pts)
    # print(f"froi: {froi}, metric: {metric}: t-value: {t_value:.2f}, p-value: {p_value:.2e}")
    # # Conduct Wilcoxon U test
    # u_stat, p_value_mwu = scipy.stats.wilcoxon(union_pts, kanwisher_pts, alternative='two-sided')
    # print(f"froi: {froi}, metric: {metric}: U-statistic: {u_stat:.2f}, p-value (Wilcoxon): {p_value_mwu:.2e}")
    # print()

    # Label fROI
    axs[fidx].text(0.95, 0.075,
                froi_labels_dict[froi],
                horizontalalignment='right', verticalalignment='bottom',
                transform=axs[fidx].transAxes, fontsize=12)

    axs[fidx].axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=1)

    axs[fidx].spines['top'].set_visible(False)
    axs[fidx].spines['right'].set_visible(False)
    axs[fidx].spines['left'].set_position(('outward', 5))
    axs[fidx].spines['bottom'].set_position(('outward', 5))
    axs[fidx].set_yticks([0, 0.25, 0.5, 0.75, 1], ['0', '25', '50', '75', '100'], fontsize=12)
    axs[fidx].spines['left'].set_bounds(0, 1)
    axs[fidx].set_ylim(-0.025, 1)
    # plt.tick_params(axis='y', labelleft=False)
    axs[fidx].spines['bottom'].set_bounds(x[0], x[-1])
    axs[fidx].set_xticks(x, [f'Split-half L1\n(Stigliani et al., 2015)', 'Intersection\n'+r'L1 $\cap$ L2', 'Split-half L2\n(Pitcher et al., 2011)'], fontsize=10)
    axs[fidx].set_xlim(x[0]-0.1, x[-1]+0.1)
    # plt.tick_params(axis='x', labelbottom=False)
    # plt.xlabel('t-threshold of model fROI lesioned', fontsize=12)
    axs[fidx].set_ylabel(f'% Intersectio over Union (IoU)', fontsize=14)
    # axs[fidx].set_xlabel(f'Localizer sets', fontsize=12)

    if froi != 'FFA':
        axs[fidx].spines['left'].set_visible(False)
        # Stop showing y-axis ticks
        axs[fidx].yaxis.set_ticks([])
        axs[fidx].set_ylabel('')

plt.tight_layout()
if savefig:
    # plt.savefig(f'{plot_dir}/lesioned_ridge_individualfroi.pdf', dpi=300, bbox_inches='tight', transparent=True)
    plt.savefig(f'{plot_dir}/figS6_IoUs.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
plt.close()

## Fig S7: Voxel-tuning matched

In [ ]:
savefig = True

plot_dir = f'{PLOTS_DIR}/supplementary_figures'
os.makedirs(plot_dir, exist_ok=True)

tthreshold = 7
frois = ['FFA', 'EBA', 'PPA']
model_localizers = ['grillspector', 'nsd']

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

all_task_optimized_model_names = ma.get_model_names(model_type='task_optimized')

subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
modelsubject_correlations_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

# frois = subjectsubject_correlations_csv['fmri_froi'].unique()
subjects = range(1, 9) # subject IDs are 1-indexed in the CSV
models = modelsubject_correlations_csv['model_name'].unique()

taskoptimized_models = [model for model in models if model in all_task_optimized_model_names]

hh_color = '#fde0c5' # Light peach
# random_color = '#f5e6d3' # Very light brown

localizer_color_dict = {
    'grillspector': '#ffffff',  # Pure white for Grill-Spector localizer
    'kanwisher': '#ce9393',            # Light red for Kanwisher localizer
    'nsd': '#B07FA0'       # Light red for NSD localizer CE8F9C
}
localizer_name_dict = {
    'grillspector': 'Category-contrast\n(L1, Stigliani et al., 2015)',
    'kanwisher': 'Kanwisher',
    'nsd': 'Voxel-matched'
}

metrics = ['normed_pearsonr', 'normed_rdmr']
lims_dict = {
    'normed_pearsonr': (0, 1.2),
    'normed_rdmr': (0, 1.4)
}
ticks_dict = {
    'normed_pearsonr': [0, 0.5, 1],
    'normed_rdmr': [0, 0.5, 1]
}

# fig, ax = plt.figure(figsize=(5, 5), dpi=300)
fig, axes = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

for midx, metric in enumerate(metrics):
    print()
    print(metric)
    for fidx, froi in enumerate(frois):
        ax = axes[midx, fidx]
        print(froi)
        hh_medians = []
        for subject in subjects:
            subjectsubject_froi = subjectsubject_correlations_csv[
                (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                (subjectsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                (
                    (subjectsubject_correlations_csv['subject1'] == subject) | 
                    (subjectsubject_correlations_csv['subject2'] == subject)
                )
            ][metric]
            hh_medians.append(subjectsubject_froi.median())
        hh_medians = np.array(hh_medians).flatten()

        # Model correlations
        model_corrs = {}
        for model_localizer in model_localizers:
            model_corrs[model_localizer] = {}

            # Separate task-optimized and brain-optimized models
            to_medians = []
            for model in taskoptimized_models:
                modelsubject_froi = modelsubject_correlations_csv[
                    (modelsubject_correlations_csv['fmri_froi'] == froi) &
                    (modelsubject_correlations_csv['model_name'] == model) &
                    (modelsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_localizer'] == model_localizer)
                ]
                model_median = modelsubject_froi[metric].median()
                # Mann-Whitney U test for each model vs. hh_medians
                # ustat, pval = scipy.stats.mannwhitneyu(model_medians, hh_medians, alternative='less')
                # if pval >= 0.05:
                #     print(f"Model {model} ({model_localizer}) is plausibly within the human population (p={pval:.2e})")
                to_medians.append(model_median)
            model_corrs[model_localizer]['task_optimized'] = np.array(to_medians)

        jitter_scale = 0.005 #0.01
        scatter_pad = 0 # 0.05
        alpha = 0.75 # 0.5
        markersize = 15 # 25

        bar_positions = np.array([0, 0.35, 0.7])

        # y=0 line
        ax.axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=0.75, zorder=0)

        # Box plot for brain-brain, and brain-ann for each model localizer
        box_data = [hh_medians]
        for model_localizer in model_localizers:
            box_data.append(model_corrs[model_localizer]['task_optimized'])
        linewidth = 0.75
        barwidth = 0.075 # 0.35
        medianlineextra = 0.075
        boxes = ax.boxplot(
                    box_data, positions=bar_positions, patch_artist=True, 
                    widths=barwidth, showfliers=False,
                    boxprops={'color': 'black', 'linewidth': linewidth},
                    whiskerprops=dict(linewidth=linewidth),
                    capprops=dict(linewidth=linewidth),
                    medianprops={'color': 'black', 'linewidth': linewidth},
                    flierprops=dict(marker='o', markersize=5, linewidth=linewidth)
                )
        for i, data in enumerate(box_data):
            med = np.median(data)
            # Manually draw a longer median line
            x = bar_positions[i]  # X-position of the boxplot
            # 0.225
            ax.plot([x - medianlineextra, x + medianlineextra], [med, med], color='black', linewidth=linewidth, solid_capstyle='butt')
        
        # Color the boxes according to category colors
        colors = [hh_color] + [localizer_color_dict[model_localizer] for model_localizer in model_localizers]
        for box, color in zip(boxes['boxes'], colors):
            box.set(facecolor=color)
        
        # Scatter individual data points
        for i, data in enumerate(box_data):
            x_ = np.zeros(len(data)) + bar_positions[i] + np.random.normal(0, jitter_scale, len(data))
            x_ += scatter_pad
            ax.scatter(
                x_, data, color='gray', alpha=alpha, zorder=5,
                edgecolors='none', s=markersize, linewidths=0.15
            )
            # print(f'Bar position {bar_positions[i]}: n={len(data)}')
        
        # Write froi name in bottom left corner
        ax.text(0.05, 0.25, froi, transform=ax.transAxes, fontsize=12, verticalalignment='top')

        yticks = np.array(ticks_dict[metric])
        ylims = lims_dict[metric]
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_position(('outward', 5))
        ax.spines['bottom'].set_position(('outward', 5))
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticks, fontsize=12)
        ax.spines['left'].set_bounds(yticks.min(), yticks.max())
        # plt.tick_params(axis='y', labelleft=False)
        ax.set_ylim(ylims)
        ax.spines['bottom'].set_bounds(bar_positions[0], bar_positions[-1])
        ax.set_xlim(bar_positions[0]-0.15, bar_positions[-1]+0.15)
        # ax.tick_params(axis='x', labelbottom=False)

        if fidx != 0:
            ax.spines['left'].set_visible(False)
            # Stop showing y-axis ticks
            ax.yaxis.set_ticks([])
        else:
            if metric == 'normed_pearsonr':
                ax.set_ylabel(r'Noise-corrected pearson $\rho$', fontsize=14)
            else:
                ax.set_ylabel(r'Noise-corrected RDM $\rho$', fontsize=14)
        # x-label on all plots for now
        # ax.set_xticks(bar_positions, ['Brain-Brain', 'Brain-ANN\n(Task-Optimized)', 'Brain-ANN\n(Brain-Optimized)'], fontsize=12)
        xtick_labels = ['Brain-Brain'] + \
            [f'Brain-ANN\n{localizer_name_dict[model_localizer]}' for model_localizer in model_localizers]
        ax.set_xticks(bar_positions)
        ax.set_xticklabels(xtick_labels, fontsize=10, rotation=0, ha='center')

plt.tight_layout()
if savefig:
    # raise NotImplementedError("Don't overwrite existing fig3_sup_otherfrois.pdf without checking it first!")
    plt.savefig(f'{plot_dir}/figS7_voxeltuningmatched.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
# plt.savefig(f'{plot_dir}/modelhuman_univariate_{froi}.pdf')
plt.close()



normed_pearsonr
FFA
EBA
PPA

normed_rdmr
FFA
EBA
PPA


## Fig S8: Other category-selective regions

In [ ]:
savefig = True

plot_dir = f'{PLOTS_DIR}/supplementary_figures'
os.makedirs(plot_dir, exist_ok=True)

tthreshold = 7
frois = ['OFA', 'FBA', 'OPA', 'VWFA', 'OVWFA']
model_localizers = ['grillspector']

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

all_task_optimized_model_names = ma.get_model_names(model_type='task_optimized')

subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
modelsubject_correlations_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

# frois = subjectsubject_correlations_csv['fmri_froi'].unique()
subjects = list(range(1, 9))
models = modelsubject_correlations_csv['model_name'].unique()

taskoptimized_models = [model for model in models if model in all_task_optimized_model_names]

hh_color = '#d24c34' # Light peach
# random_color = '#f5e6d3' # Very light brown

localizer_color_dict = {
    'grillspector': '#15a6cc',  # Light blue for Grill-Spector localizer
    'nsd': '#ce9393'            # Light red for NSD localizer
}
localizer_name_dict = {
    'grillspector': 'Category-contrast\n(L1, Stigliani et al., 2015)',
    'kanwisher': 'Kanwisher',
    'nsd': 'Voxel-matched'
}

metrics = ['normed_pearsonr', 'normed_rdmr']
lims_dict = {
    'normed_pearsonr': (0, 1.2),
    'normed_rdmr': (-0.15, 1.75)
}
ticks_dict = {
    'normed_pearsonr': [0, 0.5, 1],
    'normed_rdmr': [0, 0.5, 1, 1.5]
}

# fig, ax = plt.figure(figsize=(5, 5), dpi=300)
fig, axes = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

for midx, metric in enumerate(metrics):
    print()
    print(metric)
    for fidx, froi in enumerate(frois):
        ax = axes[midx, fidx]
        print(froi)
        hh_medians = []
        for subject in subjects:
            subjectsubject_froi = subjectsubject_correlations_csv[
                (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                (subjectsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                (
                    (subjectsubject_correlations_csv['subject1'] == subject) | 
                    (subjectsubject_correlations_csv['subject2'] == subject)
                )
            ][metric]
            hh_medians.append(subjectsubject_froi.median())
        hh_medians = np.array(hh_medians).flatten()

        # Model correlations
        model_corrs = {}
        for model_localizer in model_localizers:
            model_corrs[model_localizer] = {}

            # Separate task-optimized and brain-optimized models
            to_medians = []
            for model in taskoptimized_models:
                modelsubject_froi = modelsubject_correlations_csv[
                    (modelsubject_correlations_csv['fmri_froi'] == froi) &
                    (modelsubject_correlations_csv['model_name'] == model) &
                    (modelsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_localizer'] == model_localizer)
                ]
                model_median = modelsubject_froi[metric].median()
                # Mann-Whitney U test for each model vs. hh_medians
                # ustat, pval = scipy.stats.mannwhitneyu(model_medians, hh_medians, alternative='less')
                # if pval >= 0.05:
                #     print(f"Model {model} ({model_localizer}) is plausibly within the human population (p={pval:.2e})")
                to_medians.append(model_median)
            model_corrs[model_localizer]['task_optimized'] = np.array(to_medians)

        jitter_scale = 0.005 #0.05
        scatter_pad = 0 # 0.05
        alpha = 0.75 # 0.5
        markersize = 15 # 25

        bar_positions = np.array([0, 0.35]) #, 0.7])

        # y=0 line
        ax.axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=0.75, zorder=0)

        # Box plot for brain-brain, and brain-ann for each model localizer
        box_data = [hh_medians]
        for model_localizer in model_localizers:
            box_data.append(model_corrs[model_localizer]['task_optimized'])
        linewidth = 0.75
        barwidth = 0.075 # 0.35
        medianlineextra = 0.075
        boxes = ax.boxplot(
                    box_data, positions=bar_positions, patch_artist=True, 
                    widths=barwidth, showfliers=False,
                    boxprops={'color': 'black', 'linewidth': linewidth},
                    whiskerprops=dict(linewidth=linewidth),
                    capprops=dict(linewidth=linewidth),
                    medianprops={'color': 'black', 'linewidth': linewidth},
                    flierprops=dict(marker='o', markersize=5, linewidth=linewidth)
                )
        for i, data in enumerate(box_data):
            med = np.median(data)
            # Manually draw a longer median line
            x = bar_positions[i]  # X-position of the boxplot
            # 0.225
            ax.plot([x - medianlineextra, x + medianlineextra], [med, med], color='black', linewidth=linewidth, solid_capstyle='butt')
        
        # Color the boxes according to category colors
        colors = [hh_color] + [localizer_color_dict[model_localizer] for model_localizer in model_localizers]
        for box, color in zip(boxes['boxes'], colors):
            box.set(facecolor=color)
        
        # Scatter individual data points
        for i, data in enumerate(box_data):
            x_ = np.zeros(len(data)) + bar_positions[i] + np.random.normal(0, jitter_scale, len(data))
            x_ += scatter_pad
            ax.scatter(
                x_, data, color='gray', alpha=alpha, zorder=5,
                edgecolors='none', s=markersize, linewidths=0.15
            )
            # print(f'Bar position {bar_positions[i]}: n={len(data)}')
        
        # Write froi name in bottom left corner
        ax.text(0.05, 0.25, froi, transform=ax.transAxes, fontsize=16, verticalalignment='top')

        yticks = np.array(ticks_dict[metric])
        ylims = lims_dict[metric]
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_position(('outward', 5))
        ax.spines['bottom'].set_position(('outward', 5))
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticks, fontsize=12)
        ax.spines['left'].set_bounds(yticks.min(), yticks.max())
        # plt.tick_params(axis='y', labelleft=False)
        ax.set_ylim(ylims)
        ax.spines['bottom'].set_bounds(bar_positions[0], bar_positions[-1])
        ax.set_xlim(bar_positions[0]-0.15, bar_positions[-1]+0.15)
        # ax.tick_params(axis='x', labelbottom=False)

        if fidx != 0:
            ax.spines['left'].set_visible(False)
            # Stop showing y-axis ticks
            ax.yaxis.set_ticks([])
        else:
            if metric == 'normed_pearsonr':
                ax.set_ylabel(r'Noise-corrected pearson $\rho$', fontsize=14)
            else:
                ax.set_ylabel(r'Noise-corrected RDM $\rho$', fontsize=14)
        # x-label on all plots for now
        # ax.set_xticks(bar_positions, ['Brain-Brain', 'Brain-ANN\n(Task-Optimized)', 'Brain-ANN\n(Brain-Optimized)'], fontsize=12)
        xtick_labels = ['Brain-Brain'] + \
            [f'Brain-ANN' for model_localizer in model_localizers]
        ax.set_xticks(bar_positions)
        ax.set_xticklabels(xtick_labels, fontsize=10, rotation=0, ha='center')

plt.tight_layout()
if savefig:
    # raise NotImplementedError("Don't overwrite existing fig3_sup_otherfrois.pdf without checking it first!")
    plt.savefig(f'{plot_dir}/figS8_otherfrois.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
# plt.savefig(f'{plot_dir}/modelhuman_univariate_{froi}.pdf')
plt.close()



normed_pearsonr
OFA
FBA
OPA
VWFA
OVWFA

normed_rdmr
OFA
FBA
OPA
VWFA
OVWFA
